# Course Project - AI Engineering

**Your Name(s):** [Fill in your name(s) here]  
**Project Title:** [Fill in your project title]  
**Last Updated:** [Update this date as you work]

---

## 📋 Project Overview

This notebook is your **main workspace** for the semester-long project. You'll build and experiment with your AI system here, adding to it throughout the semester.

**Project Milestones:**
- **PM1** (Week 3): Project Proposal
- **PM2** (Week 7): MVP Prototype  
- **PM3** (Week 10): GenAI Ops Experiment
- **PM4** (Week 15): Final System & Report

**How to use this notebook:**
1. Work through sections in order
2. Fill in TODOs as you go
3. Use the Configuration section to easily change settings
4. Keep notes in markdown cells
5. Run experiments in the PM3 section

---


# 🔧 Setup (Run this first!)

This cell sets up your environment and loads helper functions.


In [ ]:
# @title Setup
!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git
import sys
sys.path.append('/content/main')

# Import course utilities
from course_utils import (
    init_openai,
    get_text_embedding,
    install_core_deps,
    seed_everything
)

# Install dependencies
install_core_deps()

# Initialize OpenAI (will prompt for API key if needed)
init_openai()

# Set random seed for reproducibility
seed_everything(42)

print('✅ Setup complete!')


installing mermaid-python
🔑 Enter your OpenAI API key.
   (It will only be stored in this Colab runtime - it's safe!)
   Get your key from: https://platform.openai.com/api-keys
OpenAI API key: ··········
✅ API key set.
✅ Setup complete!


# ⚙️ Configuration

**Edit this section to easily change settings throughout your project.**

This makes it easy to experiment with different configurations without hunting through your code.


In [ ]:
# @title Configuration Settings

# ===== RAG Settings =====
CHUNK_SIZE = 150          # Size of text chunks for RAG
CHUNK_OVERLAP = 30         # Overlap between chunks
TOP_K = 3                  # Number of chunks to retrieve

# ===== Model Settings =====
MODEL = "gpt-4o-mini"      # LLM model to use
TEMPERATURE = 0.7          # Temperature for generation (0.0-2.0)

# ===== Safety Settings =====
ENABLE_GUARDRAILS = True   # Enable safety checks
MAX_INPUT_LENGTH = 1000    # Maximum input length

# ===== Data Settings =====
# Path to your data (adjust for your project)
DATA_PATH = "/content/drive/My Drive/AI_Project/documents/"

# ===== Experiment Settings =====
# For PM3 experiments - adjust as needed
EXPERIMENT_CHUNK_SIZES = [100, 150, 200]
EXPERIMENT_TOP_KS = [1, 3, 5]

print("✅ Configuration loaded!")
print(f"   Chunk size: {CHUNK_SIZE}, Top-K: {TOP_K}, Model: {MODEL}")


✅ Configuration loaded!
   Chunk size: 150, Top-K: 3, Model: gpt-4o-mini


---

# PM1: Project Proposal (Week 3)

**Fill in this section for your project proposal.**

## Problem Statement & User

**Problem:** [Describe the problem you're solving]  
**Target Users:** [Who will use this system?]

## Why AI / LLM?

[Explain why an LLM-based system is appropriate for this problem]

## Example Interactions

**Example 1:**  
- User: "[Example query]"  
- System: "[Ideal response]"

**Example 2:**  
- User: "[Example query]"  
- System: "[Ideal response]"

**Example 3:**  
- User: "[Example query]"  
- System: "[Ideal response]"

## Data Sources

[List your data sources here, e.g., campus docs, city data, etc.]

## Initial Risks / Concerns

[What could go wrong? Hallucinations, bias, misuse, etc.]

---


---

# PM2: MVP Prototype (Week 6)

**Build a working prototype with a Gradio interface.**

## Step 1: Load Your Data


In [ ]:
# @title Load Data from Google Drive (or other source)

# Option 1: Load from Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# TODO: Update this path to match your data location
drive_path = Path("/content/drive/My Drive/AI_Project/documents/")

def load_text_files_from_drive(folder_path):
    """Load all .txt files from a Google Drive folder."""
    documents = []
    for file_path in folder_path.glob("*.txt"):
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read().strip()
            if len(text) > 50:  # Skip very short files
                documents.append(text)
    return documents

# Load your corpus
corpus = load_text_files_from_drive(drive_path)
print(f"✅ Loaded {len(corpus)} documents")

# Option 2: Use sample corpus for testing
# from course_utils import get_sample_corpus
# corpus = get_sample_corpus('mini_wiki')


## Step 2: Implement RAG Pipeline

**TODO:** Implement chunking, embedding, and retrieval functions.


In [ ]:
# @title Implement Chunking

import re
import numpy as np

def words(text):
    """Extract words from text."""
    return re.findall(r"[A-Za-z0-9']+", text.lower())

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """
    Split text into overlapping chunks.

    TODO: Implement your chunking logic here.
    Hint: Use the words() function above, then create overlapping windows.
    """
    # TODO: Your code here
    pass

# Test chunking
if len(corpus) > 0:
    test_chunks = chunk_text(corpus[0], chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    print(f"✅ Created {len(test_chunks)} chunks from first document")
    print(f"   Example chunk: {test_chunks[0][:100]}...")


NameError: name 'corpus' is not defined

In [ ]:
# @title Build Embeddings

def embed_corpus(chunks):
    """
    Embed all chunks using course_utils.get_text_embedding.

    Returns: numpy array of shape (num_chunks, embedding_dim)
    """
    # TODO: Use get_text_embedding() to embed each chunk
    pass

# Build chunks and embeddings for your corpus
all_chunks = []
for doc in corpus:
    chunks = chunk_text(doc, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    all_chunks.extend(chunks)

print(f"✅ Created {len(all_chunks)} total chunks")

# Build embeddings (this may take a minute)
print("Building embeddings...")
chunk_embeddings = embed_corpus(all_chunks)
print(f"✅ Built embeddings: shape {chunk_embeddings.shape}")

# Optional: Save embeddings to avoid recomputing
import pickle
with open('/content/drive/My Drive/AI_Project/embeddings.pkl', 'wb') as f:
    pickle.dump(chunk_embeddings, f)
print("✅ Saved embeddings to Google Drive")


In [ ]:
# @title Implement Retrieval

def normalize(v):
    """Normalize a vector to unit length."""
    v = np.array(v, dtype=np.float32)
    norm = np.linalg.norm(v)
    if norm < 1e-12:
        return v
    return v / norm

def retrieve_top_k(query, chunks, embeddings, k=TOP_K):
    """
    Retrieve top-k most similar chunks for a query.

    Uses cosine similarity (dot product of normalized vectors).
    """
    # TODO: Implement retrieval
    # 1. Embed the query
    # 2. Normalize chunk embeddings
    # 3. Compute cosine similarities
    # 4. Get top-k indices
    # 5. Return top-k chunks
    return []

# Test retrieval
if len(all_chunks) > 0:
    test_query = "What is this about?"
    retrieved = retrieve_top_k(test_query, all_chunks, chunk_embeddings, k=TOP_K)
    print(f"✅ Retrieved {len(retrieved)} chunks for query: '{test_query}'")
    print(f"   First chunk: {retrieved[0][:100]}...")


## Step 3: Generate Answers

**TODO:** Implement answer generation using retrieved context.


In [ ]:
# @title Generate Answer with RAG

from openai import OpenAI
client = OpenAI()

def generate_answer(query, context_chunks, model=MODEL, temperature=TEMPERATURE):
    """
    Generate an answer using retrieved context.

    TODO: Build a prompt that includes:
    - The user's query
    - The retrieved context chunks
    - Instructions to answer based on the context
    """
    # Build context from chunks
    # TODO: Build your prompt here
    prompt = f"""You are a helpful assistant. Answer the question based on the provided context.

Context:
{context}

Question: {query}

Answer based on the context above. If the context doesn't contain enough information, say so."""

    # Call the API
    # return result
    return None

# Test answer generation
if len(all_chunks) > 0:
    test_query = "What is this about?"
    retrieved = retrieve_top_k(test_query, all_chunks, chunk_embeddings, k=TOP_K)
    answer = generate_answer(test_query, retrieved)
    print(f"✅ Generated answer:")
    print(answer)


## Step 4: Add Safety Guardrails (PM2)

**TODO:** Add basic safety checks (refusals, input validation, etc.)


In [ ]:
# @title Safety Guardrails

def is_unsafe_query(query):
    """
    Check if query should be refused.

    TODO: Implement your safety checks here.
    Examples:
    - Medical/legal advice
    - Harmful content
    - Off-topic requests
    """
    # Example: Refuse medical advice
    # Example: Refuse legal advice
    # TODO: Add more safety checks as needed

    return False, None

def safe_rag_answer(query):
    """RAG answer with safety checks."""
    # Check safety
    # Validate input length
    # Run RAG
    return None

# Test safety
test_queries = [
    "What is the refund policy?",
    "I have chest pain, what should I do?",  # Should be refused
    "What is the weather today?"
]

for q in test_queries:
    result = safe_rag_answer(q)
    print(f"Q: {q}")
    print(f"A: {result[:100]}...")
    print()


## Step 5: Build Gradio Interface

**TODO:** Create a Gradio app for your MVP.


In [ ]:
# @title Build Gradio App

import gradio as gr

def gradio_handler(query):
    """Handler function for Gradio interface."""
    if not query or not query.strip():
        return "Please enter a question."

    answer = safe_rag_answer(query)
    return answer

# Build the interface
demo = gr.Interface(
    fn=gradio_handler,
    inputs=gr.Textbox(
        label="Your Question",
        placeholder="Ask something about your data..."
    ),
    outputs=gr.Textbox(
        label="Answer",
        lines=5
    ),
    title="[Your Project Title]",
    description="[Brief description of your project]"
)

# Launch the app
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://613312b71273ea7375.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## PM2 Status

**What works right now?**  
[Fill in what's working]

**One obvious failure mode you've observed:**  
[Fill in a failure case you've noticed]

---


---

# PM3: GenAI Ops Experiment (Week 11)

**Run three experiments using the scientific process.**

## Experiment Framework

For each experiment, you'll need:
1. **Question**: "If we change X, what happens to Y?"
2. **Hypothesis**: "We expect that..."
3. **Experiment**: What you'll change and how
4. **Metric**: How you'll measure the impact
5. **Conclusion**: What you learned

---

## Experiment 1: [Your First Experiment]


### Experiment 1: Question & Hypothesis

**Question:**  
[Example: "If we change chunk size from 100 to 300, how does answer quality change?"]

**Hypothesis:**  
[Example: "We expect that larger chunks will improve answer quality because they provide more context, but may reduce precision in retrieval."]


In [ ]:
# @title Experiment 1: Setup Test Set

# TODO: Create a test set for your experiment
# Should have 10-20 test queries with expected answers or evaluation criteria

test_set = [
    {
        "query": "Example query 1",
        "expected_keywords": ["keyword1", "keyword2"],
        "category": "category1"
    },
    # TODO: Add more test cases
]

print(f"✅ Created test set with {len(test_set)} examples")


In [ ]:
# @title Experiment 1: Run Experiment

# TODO: Implement your experiment
# Example: Compare different chunk sizes

def evaluate_answer(query, answer, expected_keywords):
    """Simple evaluation: check if key words appear in answer."""
    score = 0
    answer_lower = answer.lower()
    for keyword in expected_keywords:
        if keyword.lower() in answer_lower:
            score += 1
    return score / len(expected_keywords) if expected_keywords else 0

# Example experiment: Compare chunk sizes
results = []

for chunk_size in EXPERIMENT_CHUNK_SIZES:
    # Rebuild chunks with this chunk size
    test_chunks = []
    for doc in corpus[:5]:  # Use subset for speed
        chunks = chunk_text(doc, chunk_size=chunk_size, overlap=CHUNK_OVERLAP)
        test_chunks.extend(chunks)

    # Build embeddings
    test_embeddings = embed_corpus(test_chunks)

    # Test on each query
    scores = []
    for test_case in test_set:
        retrieved = retrieve_top_k(test_case["query"], test_chunks, test_embeddings, k=TOP_K)
        answer = generate_answer(test_case["query"], retrieved)
        score = evaluate_answer(test_case["query"], answer, test_case.get("expected_keywords", []))
        scores.append(score)

    avg_score = np.mean(scores)
    results.append({
        "chunk_size": chunk_size,
        "avg_score": avg_score,
        "scores": scores
    })
    print(f"Chunk size {chunk_size}: avg score = {avg_score:.3f}")

# Display results
import pandas as pd
df_results = pd.DataFrame(results)
print("\n📊 Results:")
print(df_results[["chunk_size", "avg_score"]])


In [ ]:
# @title Experiment 1: Visualize Results

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(df_results["chunk_size"], df_results["avg_score"], marker='o')
plt.xlabel("Chunk Size")
plt.ylabel("Average Score")
plt.title("Experiment 1: Chunk Size vs Answer Quality")
plt.grid(True)
plt.tight_layout()
plt.show()


### Experiment 1: Conclusion

**What did you find?**  
[Summarize your results]

**Did it match your hypothesis?**  
[Yes/No and why]

**What would you try next?**  
[Next steps or improvements]

---

## Experiment 2: [Your Second Experiment]

[Repeat the same structure as Experiment 1]

---

## Experiment 3: [Your Third Experiment]

[Repeat the same structure as Experiment 1]

---


---

# PM4: Final System (Week 15)

**Complete your system with all components integrated.**

## Optional: Add Agent Logic with DSPy ReAct


In [ ]:
# @title Optional: DSPy ReAct Agent

# Uncomment to use DSPy ReAct for tool use
# import dspy
# dspy.configure(lm=dspy.LM("openai/gpt-4o-mini"))

# def rag_tool(query: str) -> str:
#     """Answer questions using RAG over your documents."""
#     retrieved = retrieve_top_k(query, all_chunks, chunk_embeddings, k=TOP_K)
#     context = "\n\n".join(retrieved)
#     answer = generate_answer(query, retrieved)
#     return answer

# # Create ReAct agent
# agent = dspy.ReAct(
#     signature="question -> answer",
#     tools=[rag_tool],
#     max_iters=5
# )

# # Use it
# result = agent(question="Your question here")
# print(result.answer)


## Final Gradio App

**Build your complete Gradio app with all features.**


In [ ]:
# @title Final Gradio App

import gradio as gr

def final_handler(query):
    """Final handler with all features integrated."""
    # Safety checks
    is_unsafe, refusal_message = is_unsafe_query(query)
    if is_unsafe:
        return refusal_message

    # Input validation
    if len(query) > MAX_INPUT_LENGTH:
        return f"Query too long. Max {MAX_INPUT_LENGTH} characters."

    # RAG pipeline
    retrieved = retrieve_top_k(query, all_chunks, chunk_embeddings, k=TOP_K)
    answer = generate_answer(query, retrieved)

    # TODO: Add citations, confidence scores, or other trust signals
    return answer

# Build complete interface
with gr.Blocks(title="[Your Project Title]") as demo:
    gr.Markdown("# [Your Project Title]")
    gr.Markdown("[Brief description]")

    with gr.Row():
        with gr.Column():
            query_input = gr.Textbox(
                label="Your Question",
                placeholder="Ask something...",
                lines=3
            )
            submit_btn = gr.Button("Submit", variant="primary")

        with gr.Column():
            answer_output = gr.Textbox(
                label="Answer",
                lines=10
            )

    # Optional: Add tabs for settings, evaluation, etc.
    # with gr.Tabs():
    #     with gr.Tab("Query"):
    #         # Query interface
    #         pass
    #     with gr.Tab("Settings"):
    #         # Settings interface
    #         pass

    submit_btn.click(
        fn=final_handler,
        inputs=query_input,
        outputs=answer_output
    )

# Launch
demo.launch(share=True)


## Final Notes

**Instructions for running your project:**
1. Run the Setup cell
2. Update Configuration if needed
3. Run all cells in order
4. Launch the Gradio app

**Known limitations:**
[List any limitations or known issues]

**Future improvements:**
[What would you improve with more time?]

---

## 📝 Project Log

**Use this section to track your progress and decisions throughout the semester.**

### Week [X]: [Date]
- [What you worked on]
- [Decisions made]
- [Issues encountered]

---
